In [1]:
import pandas as pd
import json

In [2]:
def read_jsonl_data(path : str, n :int = None) -> pd.DataFrame:
    data = []
    with open(path, 'r') as f:
        for i, line in enumerate(f):
            if n is not None and i >= n:
                break
            data.append(json.loads(line))

    return pd.DataFrame(data)

In [3]:
train_py_df = read_jsonl_data('/drive1/cuongtm/ntat/Archive/sven0802/0802.train.jsonl')
val_py_df = read_jsonl_data('/drive1/cuongtm/ntat/Archive/sven0802/0802.valid.jsonl')
test_py_df = read_jsonl_data('/drive1/cuongtm/ntat/Archive/sven0802/0802.test.jsonl')

train_c_df = read_jsonl_data('/drive1/cuongtm/ntat/Archive/PrimeVul/PrimeVul/train_norm.jsonl')
val_c_df = read_jsonl_data('/drive1/cuongtm/ntat/Archive/PrimeVul/PrimeVul/val_norm.jsonl')
test_c_df = read_jsonl_data('/drive1/cuongtm/ntat/Archive/PrimeVul/PrimeVul/test_norm.jsonl')

In [4]:
c_df = pd.concat([train_c_df, val_c_df, test_c_df], ignore_index=True).drop_duplicates(subset=['code']).sample(frac=1)

In [5]:
import re

def remove_comments(text):
    # Regex này bảo vệ nội dung trong nháy đơn/kép và xóa các loại comment phổ biến
    pattern = r'(\".*?\"|\'.*?\')|(/\*.*?\*/|//[^\r\n]*|#.*)'
    def replacer(match):
        # Nếu khớp với group 2 (comment), xóa đi. Nếu khớp group 1 (chuỗi), giữ nguyên.
        return "" if match.group(2) is not None else match.group(1)
    
    return re.sub(pattern, replacer, str(text), flags=re.DOTALL)

# Áp dụng vào cột 'code'
c_df['code'] = c_df['code'].apply(remove_comments)

In [6]:
from sklearn.model_selection import train_test_split

# 1. Tách train (80%) và phần còn lại (20%)
train_c_df, temp_df = train_test_split(c_df, test_size=0.2, random_state=42)

# 2. Tách phần còn lại thành val (10%) và test (10%) - tức là chia đôi 50/50
val_c_df, test_c_df = train_test_split(temp_df, test_size=0.5, random_state=42)


In [7]:
train_py_df.columns = ['code', 'label', 'CWE_ID']
val_py_df.columns = ['code', 'label', 'CWE_ID']
test_py_df.columns = ['code', 'label', 'CWE_ID']

In [8]:
train_py_df['lang'] = 'python'
val_py_df['lang'] = 'python'
test_py_df['lang'] = 'python'

train_c_df['lang'] = 'c'
val_c_df['lang'] = 'c'
test_c_df['lang'] = 'c'

In [9]:
# Python datasets
train_py_df.to_json(
    "/drive1/cuongtm/ntat/Archive/DataWithMyFormat/sven0802/train.jsonl",
    orient="records", lines=True
)
val_py_df.to_json(
    "/drive1/cuongtm/ntat/Archive/DataWithMyFormat/sven0802/val.jsonl",
    orient="records", lines=True
)
test_py_df.to_json(
    "/drive1/cuongtm/ntat/Archive/DataWithMyFormat/sven0802/test.jsonl",
    orient="records", lines=True
)

# C datasets
train_c_df.to_json(
    "/drive1/cuongtm/ntat/Archive/DataWithMyFormat/PrimeVul/train.jsonl",
    orient="records", lines=True
)
val_c_df.to_json(
    "/drive1/cuongtm/ntat/Archive/DataWithMyFormat/PrimeVul/val.jsonl",
    orient="records", lines=True
)
test_c_df.to_json(
    "/drive1/cuongtm/ntat/Archive/DataWithMyFormat/PrimeVul/test.jsonl",
    orient="records", lines=True
)


In [10]:
train_c_vul_df = train_c_df[(train_c_df['label'] == 1) & 
                            ((train_c_df['CWE_ID'] == 'cwe-89') | (train_c_df['CWE_ID'] == 'cwe-79') | 
                             (train_c_df['CWE_ID'] == 'cwe-78') | (train_c_df['CWE_ID'] == 'cwe-22'))]

val_c_vul_df = val_c_df[(val_c_df['label'] == 1) & 
                        ((val_c_df['CWE_ID'] == 'cwe-89') | (val_c_df['CWE_ID'] == 'cwe-79') | 
                         (val_c_df['CWE_ID'] == 'cwe-78') | (val_c_df['CWE_ID'] == 'cwe-22'))]

test_c_vul_df = test_c_df[(test_c_df['label'] == 1) & 
                          ((test_c_df['CWE_ID'] == 'cwe-89') | (test_c_df['CWE_ID'] == 'cwe-79') | 
                           (test_c_df['CWE_ID'] == 'cwe-78') | (test_c_df['CWE_ID'] == 'cwe-22'))]

In [11]:
train_c_nonvul_df = train_c_df[train_c_df['label'] == 0].sample(n=2200, random_state=42)
val_c_nonvul_df = val_c_df[val_c_df['label'] == 0].sample(n=1000, random_state=42)
test_c_nonvul_df = test_c_df[test_c_df['label'] == 0].sample(n=1500, random_state=42)

In [12]:
train_df = pd.concat([train_py_df, train_c_vul_df, train_c_nonvul_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
val_df = pd.concat([val_py_df, val_c_vul_df, val_c_nonvul_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
test_df = pd.concat([test_py_df, test_c_vul_df, test_c_nonvul_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

In [13]:
train_df.to_json(
    "/drive1/cuongtm/ntat/MulVulMoe/dataset/MulVulEx/train.jsonl",
    orient="records", lines=True
)

val_df.to_json(
    "/drive1/cuongtm/ntat/MulVulMoe/dataset/MulVulEx/val.jsonl",
    orient="records", lines=True
)

test_df.to_json(
    "/drive1/cuongtm/ntat/MulVulMoe/dataset/MulVulEx/test.jsonl",
    orient="records", lines=True
)

In [14]:
train_view = read_jsonl_data('/drive1/cuongtm/ntat/Archive/DataWithMyFormat/PrimeVul/train.jsonl')

In [15]:
train_view

,code,label,CWE_ID,lang
0,static int kvm_s390_vm_set_attr(struct kvm *kv...,0,cwe-416,c
1,static int set_haschildren(const mbentry_t *mb...,0,cwe-20,c
2,"set_text_distance(gs_point *pdist, double dx,...",0,cwe-119,c
3,\n CImgDisplay& show() {\n if (is_empt...,0,cwe-787,c
4,PassRefPtr<SerializedScriptValue> SerializedSc...,0,None,c
...,...,...,...,...
179621,base::string16 AuthenticatorClientPinTapAgainS...,0,cwe-119,c
179622,explicit FFTBase(OpKernelConstruction* ctx) ...,0,cwe-703,c
179623,static void vmx_set_efer(struct kvm_vcpu *vcpu...,0,nvd-cwe-noinfo,c
179624,"makepol(WORKSTATE *state)\n{\n\tint32\t\tval,\...",0,cwe-189,c
